# Improving ICONCLASS search

We compare varioous search strategies that can be used to index the ICONCLASS data.
Comparisons are done with: 
 - BM25 fulltext indexing
 - Semantic Search with text embeddings (SBERT, Google EmbeddingGemma)
 - Semantic Search with multimodal embeddings (CLIP) 

In [59]:
import os, random
from time import time
from rich import print
from rich.progress import track
import iconclass
IC = iconclass.init()

def flatten(obj):
    yield obj
    for item in obj:
        yield from flatten(item)
    

If we index *all* the notations it is just too many, and many of the "key" notations (those that use the (+ part in the notations ) are actually only given as convenience shortcuts in the indexing process for cataloguers.

But we do want to provide access to these items in a retrieval system.

In [45]:
all = [str(n) for n in flatten(IC)]
print(len(all), len(set(all)))
all = set(all)

2294412 1359638

There is a corpus of artworks that have been described manually by humans over many decades, and from this collection we can extract a set of notations with keys that have actually been used to describe images.
While it it not perfect, it reduces the number of items considerably if we only index the "base" set of notations plus those keys that have been used.

In [49]:
used_notation_keys = set([line.strip() for line in open('used_notation_keys.txt')])
print(len(used_notation_keys), random.sample(list(used_notation_keys), 10))

18732
[
    '45F212(+52)',
    '43C943(+6)',
    '31AA25171(+933)',
    '61B2(TAERLINCK, HENDRICK)13(+2)',
    '41D222(+0)',
    '57AA645(+0)',
    '22D110(+0)',
    '43A(+12):25G3(OLIVE-TREE)(+27)',
    '31A2731(+925)',
    '25G11(+21)'
]

# Fulltext search

We use the Tantivy software library to create a fulltext index, which supports BM25.

In [64]:
from tantivy import SchemaBuilder,TextAnalyzerBuilder, Tokenizer, Filter, Index, Document

os.mkdir('tantivy_index')
LANGS = {"french":"fr", "german":"de", "italian":"it", "english":"en", "portuguese":"pt"}

schema_builder = SchemaBuilder()
schema_builder.add_text_field("notation", stored=True, tokenizer_name='raw')
for lang, lang_code in LANGS.items():
    schema_builder.add_text_field(f"label_{lang_code}", stored=False, tokenizer_name=f"{lang_code}_stem")
schema = schema_builder.build()
index = Index(schema, path='tantivy_index')

for lang, lang_code in LANGS.items():
    analyzer = (
        TextAnalyzerBuilder(Tokenizer.simple())
        .filter(Filter.remove_long(40))
        .filter(Filter.lowercase())
        .filter(Filter.stemmer(lang))
        .build()
    )
    index.register_tokenizer(f"{lang_code}_stem", analyzer)

In [65]:
writer = index.writer()
idx = 0
start_time = time()
for notation in track(all):
    if notation.find('(+') > 1 and notation not in used_notation_keys:
        continue
    n = IC[notation]
    texts = {}
    d = Document(notation=[notation])
    for lang_code in LANGS.values():
        buf = set([y for x in n.path() for y in x.keywords(lang_code)])
        buf.add(n(lang_code))
        texts[lang_code] = '\n'.join(buf)
        d.add_text(f"label_{lang_code}", texts[lang_code])
    writer.add_document(d)
    idx += 1
writer.commit()
writer.wait_merging_threads()
fulltext_duration = time() - start_time

Output()

In [71]:
searcher = index.searcher()
queries = [x.strip() for x in open('test_queries.txt')]
for q in queries:
    query = index.parse_query(q, ["label_en"])
    r = searcher.search(query, 10)
    rr = []
    for _, x in r.hits:
        n = searcher.doc(x)['notation'][0]
        rr.append(repr(IC[n]))
    print(q, rr)

portrait
[
    '48B3 portrait, self-portrait of artist',
    '48C513(+27) portrait, self-portrait of painter (+ portrait, self-portrait of artist)',
    '48C523 portrait, self-portrait of draughtsman',
    '48C23 portrait, self-portrait of sculptor',
    '48B3(+5) portrait, self-portrait of artist (+ organization ~ arts)',
    '48C613 portrait, self-portrait of photographer',
    '48C13 portrait, self-portrait of architect',
    '48C623 portrait, self-portrait of cinematographer',
    '48C42 portrait, self-portrait of monumental artist',
    '48B3(+35) portrait, self-portrait of artist (+ painting, drawing, graphic arts)'
]

portrait of a girl
[
    '11H(REMIGIUS)9 bishop Remigius (Rémi) of Rheims; possible attributes: dove with a phial of oil in its beak, 
(kneeling) maiden, phial of oil - portrait of male saint',
    '11H(NICHOLAS)9 the bishop Nicholas of Myra (or Bari); possible attributes: anchor, boat, three golden balls 
(on a book), three purses, three children in a tub, three maidens - portrait of male saint',
    '11H(CYRIACUS)9 the deacon Cyriacus of Rome; possible attributes: book of exorcism, demon under feet, dragon, 
girl (Artemia), palm-branch, sword (or axe) - portrait of male saint',
    "61B113(+2) anonymous historical persons portrayed in a group, in a group-portrait (+ all kinds of 'portrait 
historié': mythological portrait, allegorical portrait, 'Rollenporträt', etc.)",
    '61B113(+5) anonymous historical persons portrayed in a group, in a group-portrait (+ compositional variants of
portrait)',
    '48C723 portrait of a musician',
    '61B112(+5) anonymous historical persons portrayed in a double portrait (+ compositional variants of 
portrait)',
    '48C93 portrait of a writer',
    '48C723(...) portrait of a musician (with NAME of profession)',
    '61B113(+4) anonymous historical persons portrayed in a group, in a group-portrait (+ caricature ~ portrait)'
]

bridge
[
    '85(DOG ON BRIDGE) fable of the Dog on the Bridge',
    '45E51 improvised bridge, temporary bridge ~ military engineering',
    '43C7116 London Bridge and related arch and bridge games',
    '46C11221 bridge with structure, i.e. cover or buildings',
    '46C11212 cantilever bridge',
    '46C11211 suspension bridge',
    '46C112(+0) bridge (+ variant)',
    '45E511 pontoon bridge',
    '46C1122 bridge forms',
    '45E512 Bailey bridge'
]

street
[
    '44E53 street sweeping, street sweeper',
    '44F1 street-fights, riots',
    '44E53(+0) street sweeping, street sweeper (+ variant)',
    '44F13 after the street-fight',
    '44F12 before the street-fight',
    '25I1413 sloping street; street with steps or stairs',
    '25II241 village street - II - ideal landscapes',
    '44E2 street lighting',
    '25I241 village street',
    '25I1413(+0) sloping street; street with steps or stairs (+ variant)'
]

castle
[
    '33C92 castle of Love',
    '25I5 landscape with tower or castle',
    '25I5(+0) landscape with tower or castle (+ variant)',
    '43B53 (building) sand-castle',
    '41A12(+0) castle (+ variant)',
    '25II5 landscape with tower or castle - II - ideal landscapes',
    '41A9 ruin of a dwelling, house, castle, etc.',
    '41AA12 castle - AA - civic architecture: inside',
    '41A9(+0) ruin of a dwelling, house, castle, etc. (+ variant)',
    '25I5(+1) landscape with tower or castle (+ city(-scape) with figures, staffage)'
]

...

Jupyter
[]

Jupiter
[
    '92B17 specific aspects, allegorical aspects of Jupiter; Jupiter as patron',
    '24C12 Jupiter (planet)',
    '24C36 Jupiter representing tin',
    '92B18 attributes of Jupiter',
    '92B1 (story of) Jupiter (Zeus)',
    '92B18(EAGLE) attributes of Jupiter: eagle',
    '92B16 suffering, misfortune of Jupiter',
    '92B18(...) attributes of Jupiter (with NAME)',
    '92B12 love-affairs of Jupiter',
    '92B1(+0) (story of) Jupiter (Zeus) (+ variant)'
]

running and jumping
[
    '92C39111 Britomartis, running away from Minos, jumps from a cliff into the sea',
    '43C32 long-jumping, broad-jumping',
    '43C321 hop-skip-jump',
    '43C312 jumping on trampoline',
    '43C31 high-jumping (athletics, gymnastics)',
    '43C7113 running-and-hiding games',
    '71C32322 Esau runs to meet Jacob and kisses him',
    '43C71111 running (~ game)',
    '25F38(OSTRICH)(+5212) walker and runner birds: ostrich (+ running animal)',
    '43C71261 ditch-over with jumping pole (jumping game)'
]

hatred
[
    "71H1611 Jonathan warning David of Saul's hatred",
    '71D11 Joseph incurs the hatred of his brothers (Genesis 37:1-11)',
    '98B(HANNIBAL)11 in front of the altar, young Hannibal takes the oath to fight the Romans (sworn hatred)',
    '98B(HANNIBAL)11(+0) in front of the altar, young Hannibal takes the oath to fight the Romans (sworn hatred) (+
variant)'
]

jilted lover
[
    "33C22 lovers' meeting",
    '33C2 lovers; courting, flirting',
    '33C23 couple of lovers',
    "33C235 lovers' tussle",
    "33C22(+0) lovers' meeting (+ variant)",
    '33C2(+0) lovers; courting, flirting (+ variant)',
    "42D1711 lovers' casket, 'Minnekästchen'",
    '33C23(+0) couple of lovers (+ variant)',
    '33C232 (lovers) kissing each other',
    '33C234 (lovers) caressing each other'
]